# GDELT MCP Server — Capability Testing

This notebook exercises all capabilities of the **`mcp-gdelt`** server, which wraps the [GDELT DOC 2.0 API](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/).  
It is used to validate query construction, parameter handling, and response parsing implemented in `src/mcp_gdelt/`.

### What is GDELT?
The **Global Database of Events, Language, and Tone (GDELT)** monitors news media worldwide in 65+ languages and provides real-time and historical access to:
- **Article metadata** — titles, URLs, domains, languages, source countries, tone
- **Visual Knowledge Graph (VGKG)** — AI-tagged and OCR-extracted metadata from news images

### What `mcp-gdelt` provides
| Tool | Description |
|------|-------------|
| `search_articles` | Search 65-language global news index with keyword, Boolean, and date filters |
| `search_images`   | Search GDELT Visual Knowledge Graph by AI tag, caption, or OCR content |

---
**Notebook sections**
1. Environment setup
2. Direct client import
3. Article search — basic
4. Article search — Boolean operators
5. Article search — date ranges
6. Article search — sort orders
7. Image search — AI visual tags (`imagetag`)
8. Image search — web/caption tags (`imagewebtag`)
9. Image search — OCR metadata (`imageocrmeta`)
10. Multi-query comparison
11. Data analysis & visualisation

## 1 · Environment Setup
Clone the repository and install `mcp-gdelt` in editable mode so we can import the internal async client directly.

In [ ]:
# ------------------------------------------------------------------
# Clone repo and install the package (Colab-safe: skips if present)
# ------------------------------------------------------------------
import os, subprocess, sys

REPO_URL = "https://github.com/dshipley71/mcp-servers.git"
REPO_DIR = "/content/mcp-servers"
PKG_DIR  = os.path.join(REPO_DIR, "mcp-gdelt")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
else:
    print("Repository already cloned — pulling latest changes")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

# Install the package and its runtime dependencies
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", PKG_DIR],
    check=True
)
print("✓ mcp-gdelt installed")

In [ ]:
# Standard library / third-party imports used throughout the notebook
import asyncio
import json
import os
from datetime import datetime, timedelta, timezone
from collections import Counter

import httpx
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, HTML, Image as IPImage

# Raise the connect/read timeout BEFORE importing mcp_gdelt so that the
# config singleton (created at import time) picks up the higher value.
# ConnectTimeout on cold TLS to api.gdeltproject.org needs > 30 s in Colab.
os.environ.setdefault("GDELT_API_TIMEOUT", "60")

# mcp-gdelt internals
import sys
sys.path.insert(0, "/content/mcp-servers/mcp-gdelt/src")

from mcp_gdelt.services.gdelt_client import GDELTClient
from mcp_gdelt.types import (
    SearchArticlesInput,
    SearchImagesInput,
    GDELTArticle,
    GDELTImage,
)

print("✓ Imports complete")

In [ ]:
# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------

client = GDELTClient()

# GDELT enforces one request per 5 seconds.  All search calls are
# serialised through _rate_limited_call which enforces that minimum gap.
_GDELT_MIN_INTERVAL = 6.0          # seconds between requests (>5 s buffer)
_last_request_time: float = 0.0    # monotonic clock, updated AFTER each call

async def _rate_limited_call(coro_fn):
    """Wait until the GDELT min-interval has passed, then execute coro_fn."""
    global _last_request_time
    loop = asyncio.get_event_loop()
    elapsed = loop.time() - _last_request_time
    if elapsed < _GDELT_MIN_INTERVAL:
        await asyncio.sleep(_GDELT_MIN_INTERVAL - elapsed)
    result = await coro_fn()        # raises on error before we update the clock
    _last_request_time = loop.time()  # only advance clock on a completed call
    return result

def _is_rate_limit(exc: RuntimeError) -> bool:
    return "429" in str(exc) or "rate limit" in str(exc).lower()

def _retry_after(exc: RuntimeError) -> float:
    """Extract Retry-After seconds from a 429 error, or return a default."""
    import re
    m = re.search(r"Retry-After:\s*(\d+)", str(exc))
    return float(m.group(1)) + 2 if m else 30.0

async def _with_retry(coro_fn, *, retries: int = 4, base_wait: float = 6.0):
    """Execute coro_fn with rate limiting, retrying on transient failures."""
    last_exc: RuntimeError | None = None
    for attempt in range(retries):
        try:
            return await _rate_limited_call(coro_fn)
        except RuntimeError as exc:
            last_exc = exc
            if attempt < retries - 1:
                if _is_rate_limit(exc):
                    wait = _retry_after(exc)
                    print(f"  ⚠ Rate limited — waiting {wait:.0f} s (attempt {attempt + 2}/{retries})")
                else:
                    wait = base_wait * (2 ** attempt)
                    print(f"  ⚠ Attempt {attempt + 1}/{retries} failed — retrying in {wait:.0f} s: {exc}")
                await asyncio.sleep(wait)
    raise last_exc

async def search_articles(**kwargs) -> list[GDELTArticle]:
    """Thin wrapper with rate limiting + retry."""
    async def _call():
        inp = SearchArticlesInput(**kwargs)
        resp = await client.search_articles(inp)
        return resp.articles or []
    return await _with_retry(_call)

async def search_images(**kwargs) -> list[GDELTImage]:
    """Thin wrapper with rate limiting + retry."""
    async def _call():
        inp = SearchImagesInput(**kwargs)
        resp = await client.search_images(inp)
        return resp.images or []
    return await _with_retry(_call)

def articles_to_df(articles: list[GDELTArticle]) -> pd.DataFrame:
    return pd.DataFrame([a.model_dump() for a in articles])

def images_to_df(images: list[GDELTImage]) -> pd.DataFrame:
    return pd.DataFrame([i.model_dump() for i in images])

def show_articles(articles: list[GDELTArticle], n: int = 10) -> None:
    """Pretty-print the first n articles."""
    df = articles_to_df(articles)[["seendate", "domain", "language", "sourcecountry", "title"]]
    display(df.head(n).style.set_properties(**{"text-align": "left"}))

print("✓ Helper functions ready")

---
## 3 · Article Search — Basic
Demonstrate a simple single-keyword search with default parameters.

In [ ]:
articles_basic = await search_articles(
    query="climate change",
    max_records=20,
    timespan="7d",
)

print(f"Results returned: {len(articles_basic)}")
show_articles(articles_basic)

In [ ]:
# Inspect a single article object
if articles_basic:
    first = articles_basic[0]
    print(json.dumps(first.model_dump(), indent=2))

---
## 4 · Article Search — Boolean Operators
GDELT supports `AND`, `OR`, and exact phrase matching with double quotes.

In [ ]:
# Exact phrase search
articles_phrase = await search_articles(
    query='"artificial intelligence" AND regulation',
    max_records=25,
    timespan="14d",
)
print(f'Exact phrase "artificial intelligence" AND regulation: {len(articles_phrase)} results')
show_articles(articles_phrase)

In [ ]:
# OR operator — broaden the search across two topics
articles_or = await search_articles(
    query='"solar energy" OR "wind energy"',
    max_records=30,
    timespan="7d",
)
print(f'"solar energy" OR "wind energy": {len(articles_or)} results')
show_articles(articles_or)

In [ ]:
# Multi-term AND — narrow the search
articles_and = await search_articles(
    query='"electric vehicles" AND "battery" AND "charging"',
    max_records=25,
    timespan="14d",
)
print(f'EV + battery + charging: {len(articles_and)} results')
show_articles(articles_and)

---
## 5 · Article Search — Date Ranges
Use `start_date_time` / `end_date_time` (format: `YYYYMMDDHHMMSS`) for precise windows,  
or the `timespan` shorthand (`15min`, `1h`, `24h`, `7d`, `1month`, …).

In [ ]:
# Timespan shorthand — last 24 hours
articles_24h = await search_articles(
    query="earthquake",
    max_records=20,
    timespan="24h",
)
print(f"Earthquake news — last 24 h: {len(articles_24h)} results")
show_articles(articles_24h)

In [ ]:
# Absolute date range — compute a 3-day window ending now
now = datetime.now(timezone.utc)
three_days_ago = now - timedelta(days=3)

start_dt = three_days_ago.strftime("%Y%m%d%H%M%S")
end_dt   = now.strftime("%Y%m%d%H%M%S")

print(f"Window: {start_dt} → {end_dt}")

articles_range = await search_articles(
    query="cybersecurity breach",
    max_records=20,
    start_date_time=start_dt,
    end_date_time=end_dt,
)
print(f"Cybersecurity breach — 3-day window: {len(articles_range)} results")
show_articles(articles_range)

In [ ]:
# All available timespan shorthand examples
timespans = ["15min", "1h", "4h", "12h", "24h", "3d", "7d", "2weeks", "1month"]
counts = {}

for ts in timespans:
    try:
        articles = await search_articles(query="economy", max_records=1, timespan=ts)
        # max_records=1 just to test the timespan param; GDELT may return 0–1
        counts[ts] = len(articles)
        print(f"  timespan={ts:10s} → {counts[ts]} result(s) (limited to 1)")
    except Exception as exc:
        counts[ts] = -1
        print(f"  timespan={ts:10s} → ERROR: {exc}")

---
## 6 · Article Search — Sort Orders
GDELT supports five sort modes: `DateDesc`, `DateAsc`, `ToneAsc`, `ToneDesc`, `HybridRel`.

In [ ]:
sort_modes = ["DateDesc", "DateAsc", "ToneAsc", "ToneDesc", "HybridRel"]

for mode in sort_modes:
    arts = await search_articles(
        query="global health",
        max_records=5,
        timespan="7d",
        sort=mode,
    )
    print(f"\n--- sort={mode} ({len(arts)} results) ---")
    for a in arts:
        print(f"  [{a.seendate}] {a.domain}: {a.title[:80]}")

---
## 7 · Image Search — AI Visual Tags (`imagetag`)
`imagetag` uses GDELT's computer-vision pipeline to identify visual content in images.

In [ ]:
images_fire = await search_images(
    query="fire",
    max_records=20,
    timespan="7d",
    image_type="imagetag",
)
print(f"Images tagged 'fire' (AI visual): {len(images_fire)}")
display(images_to_df(images_fire).head(10))

In [ ]:
# Display the first few images inline
for img in images_fire[:4]:
    try:
        resp = httpx.get(img.url, timeout=10, follow_redirects=True)
        if resp.status_code == 200:
            display(IPImage(data=resp.content, width=300))
            print(f"  {img.url}  ({img.width}×{img.height}, {img.format})")
    except Exception as exc:
        print(f"  Could not load {img.url}: {exc}")

In [ ]:
# Try several visual tags
visual_tags = ["protest", "flood", "military", "summit", "hospital"]

tag_counts = {}
for tag in visual_tags:
    imgs = await search_images(query=tag, max_records=50, timespan="7d", image_type="imagetag")
    tag_counts[tag] = len(imgs)
    print(f"  imagetag='{tag}': {len(imgs)} images")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(tag_counts.keys(), tag_counts.values(), color="steelblue")
ax.set_title("Image count by AI visual tag (last 7 days)")
ax.set_ylabel("# images")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

---
## 8 · Image Search — Web/Caption Tags (`imagewebtag`)
`imagewebtag` searches text extracted from captions, alt text, and surrounding context.

In [ ]:
images_web = await search_images(
    query="climate summit",
    max_records=20,
    timespan="1month",
    image_type="imagewebtag",
)
print(f"imagewebtag 'climate summit': {len(images_web)} results")
display(images_to_df(images_web).head(10))

In [ ]:
# Compare imagetag vs imagewebtag for the same query
query_compare = "earthquake"

imgs_tag = await search_images(query=query_compare, max_records=50, timespan="7d", image_type="imagetag")
imgs_web = await search_images(query=query_compare, max_records=50, timespan="7d", image_type="imagewebtag")

print(f"Query: '{query_compare}'")
print(f"  imagetag    → {len(imgs_tag)} images")
print(f"  imagewebtag → {len(imgs_web)} images")

---
## 9 · Image Search — OCR Metadata (`imageocrmeta`)
`imageocrmeta` finds images whose OCR-extracted text matches the query.

In [ ]:
images_ocr = await search_images(
    query="press conference",
    max_records=20,
    timespan="14d",
    image_type="imageocrmeta",
)
print(f"imageocrmeta 'press conference': {len(images_ocr)} results")
display(images_to_df(images_ocr).head(10))

In [ ]:
# All three image types side-by-side for one query
query_all = "parliament"
image_types = ["imagetag", "imagewebtag", "imageocrmeta"]

type_counts = {}
for itype in image_types:
    imgs = await search_images(query=query_all, max_records=50, timespan="14d", image_type=itype)
    type_counts[itype] = len(imgs)
    print(f"  {itype:15s} → {len(imgs)} images")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(type_counts.keys(), type_counts.values(), color=["#2196F3", "#4CAF50", "#FF9800"])
ax.set_title(f"Image search types for '{query_all}' (last 14 days)")
ax.set_ylabel("# images")
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

---
## 10 · Multi-Query Comparison
Run several queries in parallel (using `asyncio.gather`) and compare results.

In [ ]:
topics = [
    "nuclear energy",
    "renewable energy",
    "fossil fuels",
    "hydrogen fuel",
    "carbon capture",
]

# Run sequentially — the rate limiter in search_articles enforces the
# 5 s GDELT gap, so concurrent gather() would just queue up anyway.
counts_by_topic = {}
for t in topics:
    arts = await search_articles(query=t, max_records=50, timespan="7d")
    counts_by_topic[t] = len(arts)
    print(f"  {t:25s}: {len(arts)}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(counts_by_topic.keys()), list(counts_by_topic.values()), color="teal")
ax.set_xlabel("# articles (last 7 days, max 50)")
ax.set_title("Energy topic coverage in GDELT (last 7 days)")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

---
## 11 · Data Analysis & Visualisation
Work with the article metadata to analyse language coverage, source countries, and domain diversity.

In [ ]:
# Fetch a larger result set for analysis
ANALYSIS_QUERY = "technology"
ANALYSIS_MAX   = 250

articles_analysis = await search_articles(
    query=ANALYSIS_QUERY,
    max_records=ANALYSIS_MAX,
    timespan="7d",
    sort="DateDesc",
)
df = articles_to_df(articles_analysis)
print(f"Fetched {len(df)} articles for '{ANALYSIS_QUERY}' (last 7 days)")
df.head()

In [ ]:
# ── Language distribution ──────────────────────────────────────────
lang_counts = df["language"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 4))
lang_counts.plot(kind="bar", ax=ax, color="slateblue")
ax.set_title(f"Top languages — '{ANALYSIS_QUERY}' coverage (last 7 days)")
ax.set_xlabel("Language")
ax.set_ylabel("# articles")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# ── Source country distribution ───────────────────────────────────
country_counts = df["sourcecountry"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 4))
country_counts.plot(kind="bar", ax=ax, color="darkorange")
ax.set_title(f"Top source countries — '{ANALYSIS_QUERY}' (last 7 days)")
ax.set_xlabel("Country")
ax.set_ylabel("# articles")
ax.tick_params(axis="x", rotation=45)
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# ── Top domains ───────────────────────────────────────────────────
domain_counts = df["domain"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 5))
domain_counts.plot(kind="barh", ax=ax, color="mediumseagreen")
ax.set_title(f"Top 20 domains — '{ANALYSIS_QUERY}' (last 7 days)")
ax.set_xlabel("# articles")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

In [ ]:
# ── Volume over time (hourly bins) ───────────────────────────────
if "seendate" in df.columns and df["seendate"].notna().any():
    df["seendate_parsed"] = pd.to_datetime(df["seendate"], format="%Y%m%dT%H%M%SZ", errors="coerce")
    hourly = df.set_index("seendate_parsed").resample("6h").size()

    fig, ax = plt.subplots(figsize=(12, 4))
    hourly.plot(ax=ax, color="royalblue", marker="o", markersize=4)
    ax.set_title(f"Article volume over time — '{ANALYSIS_QUERY}' (6-hour bins)")
    ax.set_ylabel("# articles")
    ax.set_xlabel("Date")
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    plt.tight_layout()
    plt.show()
else:
    print("No parseable dates in result set")

In [ ]:
# ── Summary statistics ───────────────────────────────────────────
print("=" * 55)
print(f" GDELT Query Summary — '{ANALYSIS_QUERY}'")
print("=" * 55)
print(f" Total articles     : {len(df)}")
print(f" Unique domains     : {df['domain'].nunique()}")
print(f" Unique languages   : {df['language'].nunique()}")
print(f" Unique countries   : {df['sourcecountry'].nunique()}")
if "seendate_parsed" in df.columns:
    valid_dates = df["seendate_parsed"].dropna()
    if not valid_dates.empty:
        print(f" Earliest article   : {valid_dates.min()}")
        print(f" Latest article     : {valid_dates.max()}")
print("=" * 55)

---
## 12 · Cleanup
Close the async HTTP client when the notebook session is done.

In [ ]:
await client.aclose()
print("✓ GDELTClient closed")

---
## Quick Reference

### `search_articles` parameters
| Parameter | Type | Default | Notes |
|-----------|------|---------|-------|
| `query` | str | — | Keywords, `"exact phrase"`, `AND` / `OR` |
| `max_records` | int | 50 | 1 – 250 |
| `timespan` | str | `"1month"` | `15min` · `1h` · `24h` · `7d` · `1month` |
| `sort` | str | `DateDesc` | `DateDesc` · `DateAsc` · `ToneAsc` · `ToneDesc` · `HybridRel` |
| `start_date_time` | str | — | `YYYYMMDDHHMMSS` |
| `end_date_time` | str | — | `YYYYMMDDHHMMSS` |

### `search_images` parameters
| Parameter | Type | Default | Notes |
|-----------|------|---------|-------|
| `query` | str | — | Visual concept, caption text, or OCR text |
| `max_records` | int | 50 | 1 – 250 |
| `timespan` | str | `"1month"` | Same shorthands as above |
| `image_type` | str | `imagetag` | `imagetag` · `imagewebtag` · `imageocrmeta` |

### GDELT DOC 2.0 API base URL
```
https://api.gdeltproject.org/api/v2/doc/doc
```

### Useful resources
- [GDELT DOC 2.0 API documentation](https://blog.gdeltproject.org/gdelt-doc-2-0-api-debuts/)
- [GDELT Project homepage](https://www.gdeltproject.org/)
- [mcp-gdelt README](https://github.com/dshipley71/mcp-servers/tree/main/mcp-gdelt)